## Install Libraries

Builds the embedded knowledge base (architecture §5.1): read a user-supplied KB
CSV (`CUI,concept`) from Google Drive, embed every concept name, and write the
vectors to **one** `.npz` artifact at `KB_ARTIFACT_PATH`. This notebook uses
only the shared `source/common/` package — no feedback, fusion, re-ranking or
LLM — and its artifact is consumed by `retrieval.ipynb` and `baseline.ipynb`.

The row `C0549184,NaN` is a real concept (the "none/no" answer encoded by the
source KB), so it is embedded like every other row; it is not treated as a
missing value.

Sources: `docs/architecture.md`, `paper/reference_abstract.md`,
`source/GRF/README.md`.

The encoder pin mirrors `source/GRF/requirements.txt`
(`sentence-transformers>=2.7`). `tqdm` is installed explicitly because every
long stage reports progress. numpy is left as Colab ships it — upgrading it can
break the runtime.

In [ ]:
%pip install "tqdm>=4.66" "sentence-transformers>=2.7"

## Import Libraries

Third-party imports only. The shared `common` package lives inside the
repository clone, so project modules are imported at the end of the
repository-setup stage below.

In [ ]:
from __future__ import annotations

import csv
import gc
import os
import shutil
import subprocess
import sys
import time
from contextlib import contextmanager
from pathlib import Path

import numpy as np
from tqdm.auto import tqdm

print("Python:", sys.version.split()[0])
print("numpy:", np.__version__)
print("tqdm:", __import__("tqdm").__version__)


@contextmanager
def progress_bar(total, desc: str, unit: str = "item"):
    """Drive a tqdm bar from a progress_callback(done, total)."""
    bar = tqdm(total=total, desc=desc, unit=unit)

    def report(done: int, total=None) -> None:
        if total is not None and total != bar.total:
            bar.total = total
        bar.update(done - bar.n)

    try:
        yield report
    finally:
        bar.close()

## Constants and Configuration

Every configurable value lives in this cell: Git repository, clone directory,
Google Drive paths, encoder settings, cache directories and output paths.
Cells below only reference these names.

`GIT_REPO_URL` and `GIT_BRANCH` point at this project's repository.
`DRIVE_DATA_DIR` is the single placeholder to edit (both Drive paths derive
from it); set `KB_CSV_PATH` if the KB file is not named `kb.csv`. The output
artifact is a **single file**; its `.npz` format is the repository's
`VectorIndex.save_npz`.

In [ ]:
# ---------------------------------------------------------------------------
# Git repository (REQUIRED: replace the placeholder with your repository URL)
# ---------------------------------------------------------------------------
GIT_REPO_URL = "https://github.com/minhhhduc/bio-linking-.git"
GIT_BRANCH = "main"                                             # TODO
REPO_CLONE_DIR = Path("/content/bio_linking")

# ---------------------------------------------------------------------------
# Google Drive (REQUIRED: set DRIVE_DATA_DIR below)
# ---------------------------------------------------------------------------
MOUNT_DRIVE = True
DRIVE_MOUNT_POINT = Path("/content/drive")
DRIVE_ROOT = DRIVE_MOUNT_POINT / "MyDrive"
DRIVE_DATA_DIR = DRIVE_ROOT / "REPLACE_ME"                      # TODO: set once
KB_CSV_PATH = DRIVE_DATA_DIR / "kb.csv"                         # input (CUI,concept)
KB_ARTIFACT_PATH = DRIVE_DATA_DIR / "grf_kb.npz"                # single-file output
LOCAL_ARTIFACT_TMP = Path("/content/grf_kb.npz")               # staging copy on the Colab SSD

# ---------------------------------------------------------------------------
# Encoder / index settings
# ---------------------------------------------------------------------------
ENCODER_MODEL = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
ENCODER_BATCH_SIZE = 2048       # SentenceTransformer internal batch size
ENCODE_BATCH_SIZE = 8192       # batches used by VectorIndex.from_concepts
DEVICE = "cuda"                # "cuda" (Colab GPU) or "cpu"
VERIFY_ARTIFACT = True         # reload the saved file and search one surface form

# ---------------------------------------------------------------------------
# Runtime caches and logging
# ---------------------------------------------------------------------------
CACHE_DIR = Path("/content/grf_cache")
HF_HOME = CACHE_DIR / "huggingface"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(HF_HOME))

_START = time.time()


def log(message: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')} +{time.time() - _START:6.1f}s] {message}", flush=True)


def require_configured(value, name: str):
    text = str(value)
    if "REPLACE_ME" in text:
        raise ValueError(f"{name} is still a placeholder: {text!r}. Set it in the constants cell.")
    return value


log("Configuration loaded.")
print(f"Repository      : {GIT_REPO_URL} ({GIT_BRANCH})")
print(f"KB CSV input    : {KB_CSV_PATH}")
print(f"KB artifact out : {KB_ARTIFACT_PATH}")
print(f"KB artifact tmp : {LOCAL_ARTIFACT_TMP}")
print(f"Encoder         : {ENCODER_MODEL}")

## Stage 1 — Clone or Update the GRF Repository

Colab starts from an empty runtime, so the notebook clones the code repository
and imports the shared modules from `source/common/`. Re-running this cell
updates the existing clone instead of downloading it again.

In [ ]:
repo_url = str(require_configured(GIT_REPO_URL, "GIT_REPO_URL"))


def _git(*args: str) -> None:
    log("git " + " ".join(args))
    try:
        subprocess.run(["git", *args], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(f"git {args[0]} failed with exit code {exc.returncode}.") from exc


if REPO_CLONE_DIR.exists() and not (REPO_CLONE_DIR / ".git").exists():
    raise RuntimeError(
        f"{REPO_CLONE_DIR} exists but is not a git clone. "
        "Remove it or point REPO_CLONE_DIR elsewhere."
    )

if (REPO_CLONE_DIR / ".git").exists():
    log(f"Repository already present at {REPO_CLONE_DIR}; updating branch {GIT_BRANCH!r}.")
    _git("-C", str(REPO_CLONE_DIR), "fetch", "--depth", "1", "origin", GIT_BRANCH)
    _git("-C", str(REPO_CLONE_DIR), "checkout", GIT_BRANCH)
    _git("-C", str(REPO_CLONE_DIR), "pull", "--ff-only", "origin", GIT_BRANCH)
else:
    log(f"Cloning {repo_url} (branch {GIT_BRANCH!r}) into {REPO_CLONE_DIR} ...")
    REPO_CLONE_DIR.parent.mkdir(parents=True, exist_ok=True)
    _git("clone", "--depth", "1", "--branch", GIT_BRANCH, repo_url, str(REPO_CLONE_DIR))

SOURCE_DIR = REPO_CLONE_DIR / "source"
if not (SOURCE_DIR / "common" / "__init__.py").is_file():
    raise FileNotFoundError(
        f"{SOURCE_DIR / 'common' / '__init__.py'} is missing; check GIT_BRANCH and the repository layout."
    )
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

from common.encoder import SentenceTransformerEncoder
from common.index import SINGLE_FILE_FORMAT, VectorIndex
from common.kb import load_concepts_csv, surface_entries

log(f"Repository ready at {REPO_CLONE_DIR}; common imported from {SOURCE_DIR}.")
log(f"Single-file artifact format: {SINGLE_FILE_FORMAT}")

## Stage 2 — Mount Google Drive and Validate the Input KB

The input KB is **not** in the Git repository; it comes from Google Drive. This
stage mounts Drive, checks the configured file exists, and counts its records so
later stages can report real totals.

In [ ]:
if MOUNT_DRIVE:
    try:
        from google.colab import drive
    except ImportError as exc:  # running outside Colab
        raise RuntimeError(
            "Google Colab is required to mount Drive. Run this notebook in Colab, "
            "or set MOUNT_DRIVE = False only if the inputs are already local to the runtime."
        ) from exc
    drive.mount(str(DRIVE_MOUNT_POINT))

kb_path = Path(require_configured(KB_CSV_PATH, "KB_CSV_PATH"))
if not kb_path.is_file():
    raise FileNotFoundError(f"KB CSV not found: {kb_path}")
if kb_path.suffix.lower() != ".csv":
    raise ValueError(f"KB_CSV_PATH must end in .csv, got {kb_path.name!r}.")

size_mb = kb_path.stat().st_size / 1024 ** 2
log(f"Input KB: {kb_path} ({size_mb:.2f} MB)")

with kb_path.open("r", encoding="utf-8-sig", newline="") as handle:
    reader = csv.reader(handle)
    header = next(reader, None)
    if header is None:
        raise ValueError(f"{kb_path} is empty.")
    total_records = sum(
        1 for row in tqdm(reader, desc="Counting KB rows", unit="row")
        if row and any(field.strip() for field in row)
    )
log(f"KB data rows: {total_records:,} (header: {header})")
if total_records == 0:
    raise ValueError(f"{kb_path} contains no records.")

## Stage 3 — Parse and Validate KB Concepts

`load_concepts_csv` validates the `CUI,concept` header and every row, and
raises with the offending line on bad input. A concept named `NaN` (for
example C0549184) is a legitimate KB entry — the "none/no" answer — and is
kept; only a genuinely empty field is rejected. A repeated CUI is folded into
aliases instead of aborting the load. It reports each parsed row through
`progress_callback`, which drives the bar. `surface_entries` expands concepts
into the index contract: one entry per distinct surface form, with homonyms
rewritten as `alias (preferred name)` (architecture §3, §5.1). Both loaders
report per-record progress; the expansion is reused by the embedding stage
instead of being recomputed.

In [ ]:
log("Parsing and validating KB records ...")
started = time.time()
with progress_bar(total_records, "Parsing KB records", "record") as report:
    concepts = load_concepts_csv(
        kb_path, progress_callback=report, total=total_records
    )
log(f"Parsed {len(concepts):,} concepts in {time.time() - started:.1f}s.")
if not concepts:
    raise ValueError(f"{kb_path} produced no usable concepts.")

log("Expanding concepts into index surface forms ...")
with progress_bar(len(concepts), "Expanding KB surfaces", "concept") as report:
    entries = surface_entries(concepts, progress_callback=report)
log(f"Surface forms to embed: {len(entries):,} (one index entry per surface form).")
for cui, surface, source_kb in entries[:3]:
    log(f"  example: cui={cui!r} surface={surface!r} source_kb={source_kb!r}")

## Stage 4 — Encode Surface Forms and Build the Vector Index

The encoder is loaded once from HuggingFace (the first run downloads the
weights). `VectorIndex.from_concepts` encodes the surface forms in batches and
reports each finished batch through `progress_callback`, which drives the
progress bar below.

In [ ]:
if DEVICE == "cuda":
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError(
            "DEVICE='cuda' but no CUDA device is available. In Colab: "
            "Runtime > Change runtime type > Hardware accelerator > GPU."
        )
    log(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    log("Running on CPU (DEVICE='cpu').")

log(f"Loading encoder {ENCODER_MODEL!r} on {DEVICE} (first run downloads model weights) ...")
started = time.time()
encoder = SentenceTransformerEncoder(
    ENCODER_MODEL, batch_size=ENCODER_BATCH_SIZE, device=DEVICE
)
log(f"Encoder ready: dim={encoder.dim} (loaded in {time.time() - started:.1f}s).")

with progress_bar(len(entries), "Embedding surface forms", "form") as report:
    started = time.time()
    index = VectorIndex.from_concepts(
        concepts,
        encoder,
        batch_size=ENCODE_BATCH_SIZE,
        encoder_info={"model": ENCODER_MODEL},
        progress_callback=report,
        entries=entries,
    )

elapsed = time.time() - started
log(
    f"Index built: {len(index):,} entries x {index.vectors.shape[1]} dims "
    f"(float32) in {elapsed:.1f}s."
)
assert len(index) == len(entries), (len(index), len(entries))
del entries, concepts             # the index now owns the surface strings

## Stage 5 — Save the Embedded KB as One File

The whole index (vectors, CUIs, surface forms, source-KB tags and encoder
metadata) is written to the single configured `KB_ARTIFACT_PATH` via
`VectorIndex.save_npz`. Nothing is split across extra files.

The archive is first written to `LOCAL_ARTIFACT_TMP` on the Colab SSD and then
copied to Drive in one sequential write. `np.savez_compressed` emits many small
writes, and on a Drive FUSE mount every one of them is a round trip; a 10 GB
archive written straight to Drive spends most of its time in that overhead.
One local write plus one copy is far faster, and the bytes are identical.

The encoder **weights are not embedded** — the artifact records the model id in
`encoder_info`, and `retrieval.ipynb` must load the same checkpoint to embed
queries into this vector space.


In [ ]:
artifact_path = Path(require_configured(KB_ARTIFACT_PATH, "KB_ARTIFACT_PATH"))
if artifact_path.suffix.lower() != ".npz":
    raise ValueError(f"KB_ARTIFACT_PATH must end in .npz, got {artifact_path.name!r}.")

artifact_path.parent.mkdir(parents=True, exist_ok=True)
staging_path = LOCAL_ARTIFACT_TMP
staging_path.parent.mkdir(parents=True, exist_ok=True)

log(f"Writing the artifact to local disk {staging_path} ...")
started = time.time()
index.save_npz(staging_path)
local_mb = staging_path.stat().st_size / 1024 ** 2
log(
    f"Local write done in {time.time() - started:.1f}s "
    f"({local_mb:.2f} MB, {len(index):,} entries, {index.vectors.shape[1]} dims)."
)

if staging_path.resolve() == artifact_path.resolve():
    log("KB_ARTIFACT_PATH is already local; skipping the Drive copy.")
else:
    log(f"Copying to Drive {artifact_path} (one sequential write) ...")
    started = time.time()
    shutil.copyfile(staging_path, artifact_path)
    elapsed = max(time.time() - started, 1e-6)
    copied_mb = artifact_path.stat().st_size / 1024 ** 2
    if artifact_path.stat().st_size != staging_path.stat().st_size:
        raise RuntimeError(
            f"Drive copy is {copied_mb:.2f} MB but the local artifact is "
            f"{local_mb:.2f} MB; the copy is incomplete."
        )
    log(
        f"Copied {copied_mb:.2f} MB in {elapsed:.1f}s "
        f"({copied_mb / elapsed:.1f} MB/s)."
    )


## Stage 6 — Verify the Saved Artifact

Reload the local staging copy (the bytes copied to Drive are identical, and the
size check in stage 5 proves it), compare metadata and embeddings, and run one
real index search with the exact surface form as the query. This proves the
saved artifact is self-contained (aside from the encoder checkpoint).

The reload keeps a second full copy of the vectors in RAM (~10 GB for 3.3M
rows x 768 float32), so set `VERIFY_ARTIFACT = False` on a standard 12.7 GB
Colab runtime. The artifact is already written at that point; this stage only
proves it round-trips.


In [ ]:
if VERIFY_ARTIFACT:
    log(f"Reloading the artifact from {staging_path} ...")
    reloaded = VectorIndex.load_npz(staging_path)
    assert len(reloaded) == len(index), (len(reloaded), len(index))
    assert reloaded.vectors.shape == index.vectors.shape
    assert list(reloaded.cuis) == list(index.cuis)
    assert list(reloaded.surfaces) == list(index.surfaces)
    assert np.allclose(reloaded.vectors, index.vectors, atol=1e-6)

    probe_text = index.surfaces[0]
    probe = encoder.encode([probe_text])[0]
    hits = reloaded.search(probe, 1)
    assert hits and hits[0].score > 0.999, hits
    log(
        f"Verification passed: top-1 for {probe_text!r} -> "
        f"{hits[0].cui} ({hits[0].surface!r}, cosine={hits[0].score:.4f})."
    )
    log(f"Artifact path: {artifact_path} ({artifact_path.stat().st_size / 1024 ** 2:.2f} MB)")
    log(f"Local staging copy: {staging_path} ({staging_path.stat().st_size / 1024 ** 2:.2f} MB)")
else:
    log("VERIFY_ARTIFACT is False; skipping the reload check.")

log("Offline KB build complete. Next: source/notebooks/GRF/retrieval.ipynb")

## Stage 7 — Release Memory

The artifact is written and verified, so drop the index (and its CPU/GPU
copies), the encoder and the parsed KB, then return cached CUDA memory to the
driver. Re-run stages 4–6 if you need them again.

In [ ]:
if "index" in globals():
    index.release_gpu()

for name in ("reloaded", "encoder", "index", "entries", "concepts"):
    globals().pop(name, None)
gc.collect()

try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        free_b, total_b = torch.cuda.mem_get_info()
        log(
            f"CUDA cache emptied: {free_b / 1024 ** 3:.2f}/"
            f"{total_b / 1024 ** 3:.2f} GiB free."
        )
except ImportError:
    pass

log("KB build memory released (index, encoder, parsed concepts).")